In [ ]:
import pandas as pd

# Data Wrangling

The data we use in this notebook is obtained from a study of heart disease, with contributing centers in Hungary, Switzerland and the United States. Only the data collected at the Cleveland Clinic (United States) is included here. Note that data collection was already performed in the 1980s.

In [ ]:
cleveland = pd.read_csv(
    "./heart_disease_data.csv", 
    delimiter=",", 
    header=0,
    na_values="?",
    index_col=0
)
cleveland.columns = pd.Index(
    [
        "age", # ... in years
        "sex", # 1 = male, 0 = female
        "chestpain", # type of chest pain (4 categories)
        "restbp", # resting blood pressure at admission (mmHg)
        "serumchol", # serum cholesterol (mg/dl)
        "fastbloodsugar", # fasting blood sugar >120mg/dl (1 = yes, 0 = no)
        "restecg", # resting ECG results (0 = normal, 1 = ST-T wave abnormal, 2 = probable or definite ventricular hypertrophy)
        "thalach", # maximum heart rate
        "exang", # exercise induced angina (1 = yes, 0 = no)
        "oldpeak", # ST depression induced by exercise (relative to resting state!)
        "slope", # slope of peak exercise ST segment (1 = up, 2 = flat, 3 = down)
        "ca", # number of major vessels coloured by fluorospy (range from 0 to 3)
        "thal", # 3 = normal, 6 = fixed defect, 7 = reversable defect
        "num", # diagnosis: 0 = <50% diameter narrowing, 1 = >50% diameter narrowing
    ]
)
# description of diagnosis restricted to absence of heart disease (num=0) or presence of heart disease (num={1, 2, 3, 4})
cleveland["num"] = cleveland["num"].replace({2: 1, 3: 1, 4: 1}) 
cleveland.head(5)

In [ ]:
# How many observations and variables are in the dataset?
cleveland.shape

In [ ]:
# Missing values variables
cleveland.isna().sum(axis=0)

In [ ]:
# are there any patients with all attributes present? 
(cleveland.isna().sum(axis=1) == 0).sum()

In [ ]:
# let's restrict our analysis to these complete cases
cleveland = cleveland[cleveland.isna().sum(axis=1) == 0]

In [ ]:
# how are diagnoses distributed?
cleveland["num"].value_counts()

In [ ]:
cleveland

## Exercise 1: select columns and filter patients

Task:
1. Create a DataFrame `subset` that contains only the columns `age` and `serumchol`.
2. Select all patients who are **60 years or older** and store them in `older`. How many patients are there?

Hint: in the first notebook, we selected columns with `df[["x", "y"]]` and filtered rows with `df[df["x"] > 15]`. Use `.shape` (as above) to count the patients.

In [ ]:
# 1. Select the columns
subset = # YOUR CODE HERE
subset

# 2. Patients aged 60 or older
older = # YOUR CODE HERE
older.shape

## Variable types

The first lecture on statistics introduced different types of variables to you. Do you remember the major dimensions along which variables were categorised? 

When loading data, it's important to check that you are aware of the type of each variable and represent it accordingly in python. `pandas` provides multiple ways of achieving this, either directly while loading the data using the `dtype` argument of the `read_csv()` method or by casting variable types using the `astype()` function of a `DataFrame`.

In [ ]:
# check the type of the variables
cleveland.dtypes

Note that `chestpain` is represented as `int64` when loaded by `pandas`. However, based on the description of the dataset it should contain the following categories:
- Value 1: typical angina
- Value 2: atypical angina
- Value 3: non-anginal pain
- Value 4: asymptomatic 

This suggests that this variable is a categorical variable. Let's represent it as such using `pandas`' `Categorical` data type.

In [ ]:
cleveland["chestpain"] = pd.Categorical(cleveland["chestpain"])
cleveland["chestpain"]

This data type also allows you to use the more informative labels provided by the description without a need to change the representation.

In [ ]:
cleveland["chestpain"] = cleveland["chestpain"].cat.rename_categories(
    {1: "typical angina", 2: "atypical angina", 3: "non-anginal pain", 4: "asymptomatic"}
)
cleveland["chestpain"]

This dataset contains a few more variables which are represented by integer values in the file provided (and would be treated as numerical by `pandas`) but are actually categorical. Let's update the entire dataset accordingly.

In [ ]:
# 1 = male, 0 = female
cleveland["sex"] = pd.Categorical(cleveland["sex"])
cleveland["sex"] = cleveland["sex"].cat.rename_categories(
    {1: "male", 0: "female"}
)
# fasting blood sugar >120mg/dl (1 = yes, 0 = no)
cleveland["fastbloodsugar"] = pd.Categorical(cleveland["fastbloodsugar"])
cleveland["fastbloodsugar"] = cleveland["fastbloodsugar"].cat.rename_categories(
    {1: ">120mg/dl", 0: "<120mg/dl"}
)
# resting ECG results (0 = normal, 1 = ST-T wave abnormal, 2 = probable or definite ventricular hypertrophy)
cleveland["restecg"] = pd.Categorical(cleveland["restecg"])
cleveland["restecg"] = cleveland["restecg"].cat.rename_categories(
    {0: "normal", 1: "ST-T abnormal", 2: "ventricular hypertrophy"}
)
# exercise induced angina (1 = yes, 0 = no)
cleveland["exang"] = pd.Categorical(cleveland["exang"])
cleveland["exang"] = cleveland["exang"].cat.rename_categories(
    {0: "no", 1: "yes"}
)
# 3 = normal, 6 = fixed defect, 7 = reversable defect
cleveland["thal"] = pd.Categorical(cleveland["thal"])
cleveland["thal"] = cleveland["thal"].cat.rename_categories(
    {3: "normal", 6: "fixed defect", 7: "reversable defect"}
)
# diagnosis: 0 = <50% diameter narrowing, 1 = >50% diameter narrowing
cleveland["num"] = pd.Categorical(cleveland["num"])
cleveland["num"] = cleveland["num"].cat.rename_categories(
    {0: "<50%", 1: ">50%"}
)

cleveland.head(5)

For more information on how to work with categorical data in `pandas`, check the [user guide](https://pandas.pydata.org/docs/user_guide/categorical.html).

Also, note that the column `ca`, representing the number of major vessels coloured by fluorospy has a fixed range from 0 to 3. It could be reasonable to change this column's variable type from `float` to `int` because a vessel can only be coloured or not coloured.

In [ ]:
cleveland["ca"] = cleveland["ca"].astype("int32")

## Exercise 2: convert a variable to categorical

The variable `slope` (slope of the peak exercise ST segment) is still stored as a number, but it is categorical:
- Value 1: up
- Value 2: flat
- Value 3: down

Task:
1. Convert `slope` to the `Categorical` data type and rename its categories to `"up"`, `"flat"` and `"down"`.
2. Count how many patients are in each category.

Hint: look at how we converted `chestpain` and `sex` above. You only need to change the column name and the labels. To count, use `value_counts()` (as for `num` above).

In [ ]:
# 1. Convert slope to categorical and rename the categories
# YOUR CODE HERE

# 2. Count the patients per category
# YOUR CODE HERE